# 캡션 생성: 나머지 ~6,010개 영상

기존 1,000개 영상은 `msrvtt_1000_captions.json`에 GPT-4o-mini 영문 캡션이 있음.  
이 노트북은 **나머지 ~6,010개** 영상에 동일한 depth의 영문 캡션을 생성한다.

- **모델**: GPT-4o-mini (Vision)
- **프레임**: 8장 균등 추출, 512×288, JPEG 85%, detail=low
- **프롬프트**: 기존 1,000개 생성 시 사용한 것과 동일
- **체크포인트**: 50개마다 중간 저장 → Colab 세션 끊겨도 resume 가능
- **최종 출력**: `msrvtt_full_captions.json` + `msrvtt_metadata.csv` 갱신

### 실행 전제
- 00_setup.ipynb가 정상 실행된 환경 (Drive 마운트, 프로젝트 클론 완료)
- MSR-VTT.ZIP이 Google Drive에 존재

In [ ]:
# ── Step 0: 환경 로드 (00_setup.ipynb 실행) ──
import os, nbformat
from IPython import get_ipython

SETUP_PATH = '/content/videorag_prototype/notebooks/00_setup.ipynb'

# 레포가 없으면 먼저 클론
if not os.path.exists('/content/videorag_prototype'):
    from google.colab import userdata
    # [Public repo] token 불필요 — public clone
    os.system("git clone https://github.com/LimPark996/VideoRAG-Public.git /content/videorag_prototype")
    print('✓ git clone 완료')

# 00_setup.ipynb 전체 실행 (경로 CONFIG, Drive 마운트 등)
with open(SETUP_PATH, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

ip = get_ipython()
for cell in nb.cells:
    if cell.cell_type == 'code':
        ip.run_cell(cell.source)

print(f'\n{"="*60}')
print(f'01b: 환경 로드 완료')
print(f'  VIDEO_DIR: {VIDEO_DIR}')
print(f'{"="*60}')

In [ ]:
# ── Step 1: 영상 압축 해제 & 전체 영상 링크 ──
import os, zipfile, shutil

ZIP_PATH         = '/content/drive/MyDrive/videorag_prototype/data/msrvtt/videos/data/MSR-VTT.ZIP'
VIDEO_DIR_ACTUAL = '/content/msrvtt/MSR-VTT/train_val_videos/TrainValVideo'

# 압축 해제
if not (os.path.exists(VIDEO_DIR_ACTUAL) and len(os.listdir(VIDEO_DIR_ACTUAL)) > 100):
    print('압축 해제 중...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/msrvtt')
print(f'✓ 전체 영상: {len(os.listdir(VIDEO_DIR_ACTUAL))}개')

# 프로젝트 VIDEO_DIR에 심볼릭 링크
os.makedirs(VIDEO_DIR, exist_ok=True)

all_video_files = sorted([f for f in os.listdir(VIDEO_DIR_ACTUAL) if f.endswith('.mp4')])
copied, skipped = 0, 0
for vf in all_video_files:
    src = os.path.join(VIDEO_DIR_ACTUAL, vf)
    dst = os.path.join(VIDEO_DIR, vf)
    if not os.path.exists(dst):
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)
        copied += 1
    else:
        skipped += 1

print(f'✓ 링크/복사: {copied}개 신규, {skipped}개 이미 존재 → {VIDEO_DIR}')
print(f'✓ VIDEO_DIR 영상 수: {len(os.listdir(VIDEO_DIR))}개')

In [ ]:
# ── Step 2: 기존 캡션 로드 & 미생성 목록 구성 ──
import json, os

!pip install -q openai

from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

DRIVE_CAPTIONS = '/content/drive/MyDrive/videorag_prototype/data/msrvtt/videos/msrvtt_captions'

# 기존 1000개 캡션 로드
CAPTION_1K_JSON = os.path.join(DRIVE_CAPTIONS, 'msrvtt_1000_captions.json')
with open(CAPTION_1K_JSON, 'r', encoding='utf-8') as f:
    raw_1k = json.load(f)

existing_captions = {}
for entry in raw_1k.values():
    vid = entry.get('video_id', '')
    cap = entry.get('caption', '')
    if vid and cap:
        existing_captions[vid] = cap

print(f'✓ 기존 캡션: {len(existing_captions)}개')

# 기존 캡션 샘플 (depth 참고)
print('\n── 기존 캡션 샘플 (depth 참고) ──')
for vid, cap in list(existing_captions.items())[:5]:
    print(f'  {vid}: {cap}')

# 전체 영상 목록
all_videos = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(VIDEO_DIR)
    if f.endswith(('.mp4', '.avi', '.mkv'))
])

# 체크포인트 로드 (이전 실행에서 중단된 경우)
CHECKPOINT_PATH = os.path.join(DRIVE_CAPTIONS, 'captions_remaining_checkpoint.json')
checkpoint_captions = {}
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        checkpoint_captions = json.load(f)
    print(f'✓ 체크포인트 로드: {len(checkpoint_captions)}개 이미 완료')

# 남은 영상 계산
done_ids = set(existing_captions.keys()) | set(checkpoint_captions.keys())
remaining = [vid for vid in all_videos if vid not in done_ids]

print(f'\n전체 영상:     {len(all_videos)}개')
print(f'캡션 완료:     {len(done_ids)}개 (기존 {len(existing_captions)} + 체크포인트 {len(checkpoint_captions)})')
print(f'남은 영상:     {len(remaining)}개')

est_cost = len(remaining) * 0.0012
est_time_min = len(remaining) * 1.5 / 60
print(f'\n예상 API 비용: ~${est_cost:.2f}')
print(f'예상 소요 시간: ~{est_time_min:.0f}분 ({est_time_min/60:.1f}시간)')

In [ ]:
# ── Step 3: 캡션 생성 (GPT-4o-mini Vision) ──
import cv2
import base64
import time
import numpy as np
from openai import OpenAI
from tqdm.notebook import tqdm

client = OpenAI()

# ── 프레임 추출 설정 (기존 1000개와 동일) ──
N_FRAMES     = 8
FRAME_W      = 512
FRAME_H      = 288
JPEG_QUALITY = 85

# ── 프롬프트 (기존 1000개 생성 시 사용한 것과 동일) ──
SYSTEM_PROMPT = (
    "You are an expert video analyst. "
    "Analyze the provided video frames and write a caption."
)

USER_PROMPT = (
    "These are 8 frames sampled from a short video clip. "
    "Write one detailed English sentence describing what happens. "
    "Focus on: main subjects, actions, setting, notable details. "
    "Do not start with 'The video shows' or 'In the video'."
)


def extract_frames_b64(video_path):
    """영상에서 8프레임 균등 추출 → base64 리스트"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    indices = np.linspace(0, total - 1, N_FRAMES, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.resize(frame, (FRAME_W, FRAME_H))
        _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        frames.append(base64.b64encode(buf).decode('utf-8'))
    cap.release()
    return frames


def generate_caption(video_path, max_retries=5):
    """GPT-4o-mini Vision으로 영문 캡션 생성"""
    frames = extract_frames_b64(video_path)
    if not frames:
        return None

    image_contents = [
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{b64}", "detail": "low"}
        }
        for b64 in frames
    ]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": USER_PROMPT},
                *image_contents
            ]
        }
    ]

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                max_tokens=150,
                temperature=0.3,
            )
            caption = resp.choices[0].message.content.strip().strip("'\"")
            return caption
        except Exception as e:
            err = str(e)
            if '429' in err:
                wait = min(60 * (attempt + 1), 300)
                print(f'  ⚠ Rate limit, {wait}초 대기...')
                time.sleep(wait)
            else:
                print(f'  ✗ 에러: {err}')
                return None
    return None


# ── 메인 루프 ──
def save_checkpoint(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

new_captions = dict(checkpoint_captions)  # 체크포인트에서 이어서
fail_ids = []
SLEEP_SEC = 0.5

print(f'캡션 생성 시작: {len(remaining)}개 영상')
print(f'체크포인트: {CHECKPOINT_PATH}')
print('=' * 60)

for i, vid in enumerate(tqdm(remaining, desc='캡션 생성')):
    video_path = os.path.join(VIDEO_DIR, f'{vid}.mp4')
    if not os.path.exists(video_path):
        fail_ids.append(vid)
        continue

    caption = generate_caption(video_path)
    if caption:
        new_captions[vid] = caption
    else:
        new_captions[vid] = f'[FAILED] {vid}'
        fail_ids.append(vid)

    # 50개마다 체크포인트
    if (i + 1) % 50 == 0:
        save_checkpoint(new_captions, CHECKPOINT_PATH)
        success = sum(1 for v in new_captions.values() if not v.startswith('[FAILED]'))
        tqdm.write(f'  체크포인트 저장: {success}/{len(new_captions)}개 성공')

    time.sleep(SLEEP_SEC)

save_checkpoint(new_captions, CHECKPOINT_PATH)

success_count = sum(1 for v in new_captions.values() if not v.startswith('[FAILED]'))
print(f'\n완료: {success_count}개 성공, {len(fail_ids)}개 실패')
if fail_ids:
    print(f'실패 목록 (상위 10개): {fail_ids[:10]}')

In [ ]:
# ── Step 4: 전체 캡션 병합 & 저장 ──
import json, os, pandas as pd

# 기존 1000개 + 신규 캡션 병합
full_captions = dict(existing_captions)
for vid, cap in new_captions.items():
    if not cap.startswith('[FAILED]'):
        full_captions[vid] = cap

print(f'전체 캡션: {len(full_captions)}개')

# 전체 캡션 JSON 저장 (Drive)
FULL_CAPTION_JSON = os.path.join(DRIVE_CAPTIONS, 'msrvtt_full_captions.json')
with open(FULL_CAPTION_JSON, 'w', encoding='utf-8') as f:
    json.dump(full_captions, f, ensure_ascii=False, indent=2)
print(f'✓ 전체 캡션 JSON: {FULL_CAPTION_JSON}')

# msrvtt_metadata.csv 갱신
METADATA_CSV = os.path.join(DATA_DIR, 'msrvtt_metadata.csv')

all_videos = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(VIDEO_DIR)
    if f.endswith(('.mp4', '.avi', '.mkv'))
])

rows = []
for vid in all_videos:
    rows.append({
        'video_id': vid,
        'caption': full_captions.get(vid, ''),
        'video_path': os.path.join(VIDEO_DIR, f'{vid}.mp4'),
    })

df = pd.DataFrame(rows)
df.to_csv(METADATA_CSV, index=False)

has_caption = df['caption'].astype(bool).sum()
no_caption  = len(df) - has_caption
print(f'✓ CSV 저장: {METADATA_CSV}')
print(f'  캡션 있음: {has_caption}개, 캡션 없음: {no_caption}개')

# Drive 백업
DRIVE_CSV = os.path.join(DRIVE_CAPTIONS, 'msrvtt_metadata_full.csv')
df.to_csv(DRIVE_CSV, index=False)
print(f'✓ Drive 백업: {DRIVE_CSV}')

In [ ]:
# ── Step 5: 캡션 품질 검증 ──
import json, random

with open(FULL_CAPTION_JSON, 'r', encoding='utf-8') as f:
    caps = json.load(f)

lengths = [len(c.split()) for c in caps.values()]
print(f'전체 캡션 수: {len(caps)}개')
print(f'평균 단어 수: {sum(lengths)/len(lengths):.1f}')
print(f'최소 단어 수: {min(lengths)}')
print(f'최대 단어 수: {max(lengths)}')

# 기존 vs 신규 비교
old_lengths = [len(c.split()) for vid, c in caps.items() if vid in existing_captions]
new_lengths = [len(c.split()) for vid, c in caps.items() if vid not in existing_captions]

print(f'\n── Depth 비교 ──')
print(f'기존 1000개:  평균 {sum(old_lengths)/max(len(old_lengths),1):.1f} 단어')
print(f'신규 ~6010개: 평균 {sum(new_lengths)/max(len(new_lengths),1):.1f} 단어')

random.seed(42)
old_samples = random.sample(list(existing_captions.keys()), min(3, len(existing_captions)))
new_only = [vid for vid in caps if vid not in existing_captions]
new_samples = random.sample(new_only, min(3, len(new_only)))

print(f'\n── 기존 캡션 샘플 ──')
for vid in old_samples:
    print(f'  [{vid}] {caps[vid]}')

print(f'\n── 신규 캡션 샘플 ──')
for vid in new_samples:
    print(f'  [{vid}] {caps[vid]}')